# 02 · Synthetic scenario and route overlap review

Run this after notebook 01. It creates the aggregate-calibrated synthetic route/hour scenario, then builds a stop-overlap review shortlist. The allocation is not measured ridership, and stop overlap is not proof of duplicate service.

Review route direction, endpoint aliases, and stop sequence before interpreting any overlap pair. This notebook does not recommend automatic route merging or service cuts.

## Section 1 · Synthetic route/hour scenario

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.gtfs import resolve_gtfs_dir, load_gtfs

GTFS_DIR = resolve_gtfs_dir(PROJECT_ROOT)
feed = load_gtfs(GTFS_DIR)
print(f"Feed folder: {GTFS_DIR}")
from src.gtfs import active_service_ids, first_trip_departures

SERVICE_DATE = "2026-10-26"  # Must be active in calendar.txt
# MTC performance page reported 32.24 lakh passengers/day for FY 2024-25.
# This is an all-network average used only as an illustrative calibration target.
# Edit this number and the year/source below if your mentor selects a different baseline.
REFERENCE_DAILY_PASSENGERS = 3_224_000
REFERENCE_SOURCE = "MTC Performance Indicators, FY 2024-25; aggregate Chennai network daily average"
REFERENCE_URL = "https://mtcbus.tn.gov.in/Home/performance"

# Peak-hour scenario multipliers (assumptions, not learned from passenger data).
PEAK_HOURS = {7, 8, 9, 17, 18, 19}
PEAK_MULTIPLIER = 1.6
OFF_PEAK_MULTIPLIER = 1.0


### 1. Build scheduled route-hour supply for the selected date

In [ ]:
active_ids = active_service_ids(feed["calendar"], SERVICE_DATE)
if not active_ids:
    raise ValueError("No calendar service active on SERVICE_DATE. Choose a covered operating date.")

departures = first_trip_departures(feed["stop_times"])
trips = feed["trips"].copy()
trips["service_id"] = trips["service_id"].astype(str)
trips = trips[trips["service_id"].isin(active_ids)].merge(departures, on="trip_id", how="left", validate="one_to_one")
trips = trips.merge(feed["routes"], on="route_id", how="left", validate="many_to_one").dropna(subset=["departure_hour"])
trips["departure_hour"] = trips["departure_hour"].astype(int)
route_hour = (trips.groupby(["route_id", "route_short_name", "route_long_name", "departure_hour"], dropna=False)
              .size().rename("scheduled_trip_count").reset_index())
if route_hour.empty:
    raise ValueError("No trips with a valid first-stop departure time were found.")
display(route_hour.head())

### 2. Allocate the aggregate total, preserving the total exactly

In [ ]:
route_hour["scenario_multiplier"] = route_hour["departure_hour"].map(
    lambda hour: PEAK_MULTIPLIER if hour in PEAK_HOURS else OFF_PEAK_MULTIPLIER
)
route_hour["allocation_weight"] = route_hour["scheduled_trip_count"] * route_hour["scenario_multiplier"]
route_hour["unrounded_estimate"] = (
    REFERENCE_DAILY_PASSENGERS * route_hour["allocation_weight"] / route_hour["allocation_weight"].sum()
)
# Largest-remainder rounding: integer counts that sum exactly to the selected daily total.
route_hour["synthetic_ridership_estimate"] = route_hour["unrounded_estimate"].astype(int)
remainder = int(REFERENCE_DAILY_PASSENGERS - route_hour["synthetic_ridership_estimate"].sum())
if remainder > 0:
    top_fractional = (route_hour["unrounded_estimate"] - route_hour["synthetic_ridership_estimate"]).nlargest(remainder).index
    route_hour.loc[top_fractional, "synthetic_ridership_estimate"] += 1
route_hour["data_status"] = "synthetic_scenario_not_observed"
route_hour["reference_daily_total"] = REFERENCE_DAILY_PASSENGERS
route_hour["reference_source"] = REFERENCE_SOURCE

assert int(route_hour["synthetic_ridership_estimate"].sum()) == REFERENCE_DAILY_PASSENGERS
print("Estimated total:", f"{route_hour['synthetic_ridership_estimate'].sum():,}")
print("Calibration total:", f"{REFERENCE_DAILY_PASSENGERS:,}")
display(route_hour.head(10))

### 3. Inspect route totals and the synthetic hourly profile

In [ ]:
route_estimates = (route_hour.groupby(["route_id", "route_short_name", "route_long_name"], dropna=False)
                   .agg(synthetic_daily_estimate=("synthetic_ridership_estimate", "sum"),
                        scheduled_trips=("scheduled_trip_count", "sum"))
                   .sort_values("synthetic_daily_estimate", ascending=False).reset_index())
display(route_estimates.head(20))

hour_profile = route_hour.groupby("departure_hour")["synthetic_ridership_estimate"].sum().reindex(range(24), fill_value=0)
ax = hour_profile.plot(kind="bar", figsize=(12, 4), color="#f18f01", title="Synthetic daily ridership allocation by scheduled departure hour")
ax.set_xlabel("Departure hour (local feed time)")
ax.set_ylabel("Synthetic passenger estimate")
ax.grid(axis="y", alpha=.25)


### 4. Save with provenance labels

In [ ]:
output_dir = PROJECT_ROOT / "dataset" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"synthetic_route_hour_ridership_{SERVICE_DATE}.csv"
route_hour.drop(columns=["unrounded_estimate"]).to_csv(output_path, index=False)
print(f"Wrote {output_path}")
print("Keep this file labeled synthetic in every chart, model, and report.")

### Model limitations and next steps

1. The daily total is a network-wide calibration target; it does not reveal how riders are split between routes.
2. Allocating in proportion to scheduled trips assumes each trip has similar demand before time-of-day multipliers. That assumption is unverified.
3. The multiplier is illustrative. Test alternatives (for example, all hours equal; moderate peaks; stronger peaks) and compare how rankings change.
4. “Route-hour ridership” here means an allocated count attached to the trip's first-stop departure hour. It is not stop-level boarding, load on a segment, or passengers simultaneously on a bus.
5. Without vehicle capacity and observed passenger loads, do not claim a route is overcrowded. The project can report **scenario-based demand pressure** only, clearly defined by the selected assumptions.
6. For stronger estimates, collect route/time-band counts with a small field survey or ask MTC/CUMTA for anonymized ticketing counts. Record date, route, direction, time band, count definition, source, and sampling method. Validate the synthetic scenario against real counts before using predictive labels.

## Section 2 · Route stop-overlap candidates

In [ ]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.sparse import csr_matrix

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.gtfs import resolve_gtfs_dir

GTFS_DIR = resolve_gtfs_dir(PROJECT_ROOT)
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
feature_files = sorted(PROCESSED_DIR.glob("route_features_*.csv"))
if not feature_files:
    raise FileNotFoundError("Run notebook 01 first; route_features_*.csv was not found.")
feature_path = feature_files[-1]
SERVICE_DATE = feature_path.stem.rsplit("_", 1)[-1]
features = pd.read_csv(feature_path, dtype={"route_id": str})
print(f"Service date: {SERVICE_DATE}; active routes: {features.route_id.nunique():,}")

### 1. Build route-to-stop membership from the supplied feed

In [ ]:
trips = pd.read_csv(GTFS_DIR / "trips.txt", dtype=str, usecols=["trip_id", "route_id"]).drop_duplicates("trip_id")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt", dtype=str, usecols=["trip_id", "stop_id"])
trip_stops = stop_times.drop_duplicates(["trip_id", "stop_id"]).merge(
    trips, on="trip_id", how="inner", validate="many_to_one")
active_route_ids = set(features["route_id"].astype(str))
route_stops = (trip_stops[trip_stops["route_id"].isin(active_route_ids)]
               .drop_duplicates(["route_id", "stop_id"])
               [["route_id", "stop_id"]])
route_order = features["route_id"].astype(str).tolist()
stop_order = sorted(route_stops["stop_id"].astype(str).unique())
route_index = {route_id:i for i,route_id in enumerate(route_order)}
stop_index = {stop_id:i for i,stop_id in enumerate(stop_order)}
row_ix = route_stops["route_id"].map(route_index).to_numpy()
col_ix = route_stops["stop_id"].map(stop_index).to_numpy()
incidence = csr_matrix((np.ones(len(route_stops),dtype=np.int32),(row_ix,col_ix)),
                       shape=(len(route_order),len(stop_order)),dtype=np.int32)
stop_counts = np.asarray(incidence.sum(axis=1)).ravel().astype(int)
print(f"Route-stop memberships: {incidence.nnz:,}; distinct stops: {len(stop_order):,}")
print("Median stops per active route:", float(np.median(stop_counts)))

### 2. Score pairs by shared stop coverage

In [ ]:
# Pairwise sparse multiplication counts shared stops without materializing route sequences.
MIN_SHARED_STOPS = 8
shared = (incidence @ incidence.T).tocoo()
mask = (shared.row < shared.col) & (shared.data >= MIN_SHARED_STOPS)
left_ix, right_ix, common = shared.row[mask], shared.col[mask], shared.data[mask].astype(int)
left_n, right_n = stop_counts[left_ix], stop_counts[right_ix]
small_n, large_n = np.minimum(left_n,right_n), np.maximum(left_n,right_n)
union = left_n + right_n - common
pairs = pd.DataFrame({
    "route_a_id": np.asarray(route_order,dtype=object)[left_ix],
    "route_b_id": np.asarray(route_order,dtype=object)[right_ix],
    "shared_stop_count": common,
    "route_a_stop_count": left_n,
    "route_b_stop_count": right_n,
    "smaller_route_stop_coverage": common / np.maximum(small_n,1),
    "larger_route_stop_coverage": common / np.maximum(large_n,1),
    "stop_set_jaccard": common / np.maximum(union,1),
})
MIN_SMALLER_ROUTE_COVERAGE = 0.90
MIN_JACCARD = 0.70
MAX_REVIEW_PAIRS = 250
candidates = pairs[(pairs["smaller_route_stop_coverage"] >= MIN_SMALLER_ROUTE_COVERAGE)
                   & (pairs["stop_set_jaccard"] >= MIN_JACCARD)].copy()
filtered_pair_count = len(candidates)
candidates["review_rank_score"] = (0.6*candidates["smaller_route_stop_coverage"]
                                   + 0.4*candidates["stop_set_jaccard"])
candidates = candidates.sort_values(["review_rank_score","shared_stop_count"],ascending=False)
print(f"Pairs sharing at least {MIN_SHARED_STOPS} stops: {len(pairs):,}")
print(f"Pairs passing overlap thresholds: {filtered_pair_count:,}")
print(f"Top review pairs retained: {min(MAX_REVIEW_PAIRS,filtered_pair_count):,}")

### 3. Add route names, service supply, and direction clues

In [ ]:
route_info = features[["route_id","route_short_name","route_long_name","scheduled_trips","scheduled_hours","distinct_stop_count"]].copy()
route_info["route_id"] = route_info["route_id"].astype(str)
route_info = route_info.drop_duplicates("route_id").set_index("route_id")
for side in ["a","b"]:
    for col in ["route_short_name","route_long_name","scheduled_trips","scheduled_hours","distinct_stop_count"]:
        candidates[f"{col}_{side}"] = candidates[f"route_{side}_id"].map(route_info[col])

# Reverse endpoint matches often represent opposite directions and should not be treated as duplicate routes.
def endpoints(name):
    parts = re.split(r"\s+TO\s+", str(name), maxsplit=1, flags=re.IGNORECASE)
    return tuple(part.strip().casefold() for part in parts) if len(parts)==2 else None

def pair_type(row):
    a, b = endpoints(row["route_long_name_a"]), endpoints(row["route_long_name_b"])
    if a and b and a == (b[1], b[0]):
        return "reverse_direction_pair_review_both_directions_may_be_needed"
    if str(row["route_long_name_a"]).casefold() == str(row["route_long_name_b"]).casefold():
        return "same_name_overlap_review"
    return "shared_stop_overlap_review"

candidates["review_category"] = candidates.apply(pair_type,axis=1)
candidates["lower_trip_supply_route_id"] = np.where(
    candidates["scheduled_trips_a"] <= candidates["scheduled_trips_b"],
    candidates["route_a_id"],candidates["route_b_id"])
candidates["review_status"] = "requires_map_and_observed_demand_review"
candidates = candidates.sort_values(["review_rank_score","shared_stop_count"],ascending=False).head(MAX_REVIEW_PAIRS)
display(candidates[["route_a_id","route_short_name_a","route_long_name_a","scheduled_trips_a",
                    "route_b_id","route_short_name_b","route_long_name_b","scheduled_trips_b",
                    "shared_stop_count","smaller_route_stop_coverage","stop_set_jaccard","review_category"]].head(30))

### 4. Save the planner-review shortlist

In [ ]:
output_path = PROCESSED_DIR / f"route_overlap_candidates_{SERVICE_DATE}.csv"
candidates.to_csv(output_path,index=False)
print(f"Saved {len(candidates):,} candidates to {output_path}")
if not candidates.empty:
    print("Review categories:")
    display(candidates["review_category"].value_counts().rename_axis("category").to_frame("candidate_pairs"))
else:
    print("No pairs passed these thresholds. Adjust thresholds only with a documented reason.")

### How to use this output

Review the strongest pairs on a map and compare stop sequence and direction before discussing service coordination. A reverse-direction pair may be required to provide both directions of travel. A route with fewer scheduled trips is not necessarily underused; the feed contains no passenger counts. Do not use this file to automatically merge, cancel, or reduce a service.